In [ ]:
!pip install requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

# Kazımak istediğiniz Wikipedia makalesinin URL'si
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"

# Wikipedia'nın isteğimizi engellememesi için bir User-Agent tanımlıyoruz
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    # HTML içeriğini parse ediyoruz
    soup = BeautifulSoup(response.content, 'html.parser')

    # Wikipedia'da ana makale içeriği genellikle bu id'ye sahip div içinde yer alır
    content_div = soup.find(id='mw-content-text')

    # Div içindeki tüm paragraf (<p>) etiketlerini buluyoruz
    paragraphs = content_div.find_all('p')

    cleaned_paragraphs = []
    for p in paragraphs:
        text = p.get_text().strip()

        # Düzenli ifadeler (Regex) ile [1], [2], [citation needed] gibi ifadeleri temizliyoruz
        text = re.sub(r'\[\d+\]', '', text)
        text = re.sub(r'\[citation needed\]', '', text)

        # Çok kısa veya boş paragrafları eliyoruz (kaliteli veri için)
        if len(text) > 40:
            cleaned_paragraphs.append(text)

    # Tüm paragrafları aralarında boşluk bırakarak birleştiriyoruz
    scraped_text = "\n\n".join(cleaned_paragraphs)

    # Veriyi bir metin dosyasına kaydedelim
    with open("scraped_wikipedia.txt", "w", encoding="utf-8") as f:
        f.write(scraped_text)

    print(f"Kazıma başarılı. Toplam paragraf sayısı: {len(cleaned_paragraphs)}")
    print(f"Toplam karakter sayısı: {len(scraped_text)}")
else:
    print(f"Hata oluştu. Durum kodu: {response.status_code}")

Kazıma başarılı. Toplam paragraf sayısı: 181
Toplam karakter sayısı: 83592


In [ ]:
!pip install datasets

In [ ]:

from datasets import Dataset

# Kaydettiğimiz dosyayı okuyoruz
with open("scraped_wikipedia.txt", "r", encoding="utf-8") as f:
    paragraphs = [line.strip() for line in f.read().split("\n\n") if line.strip()]

# Hugging Face formatına dönüştürme
dataset = Dataset.from_dict({"text": paragraphs})
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 181
})


In [ ]:
!pip install transformers peft trl accelerate bitsandbytes torch

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig

model_id = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=50,
    fp16=True,
    save_strategy="no"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=training_args,
    peft_config=peft_config
)

print("Eğitim başlatılıyor...")
trainer.train()

trainer.model.save_pretrained("./scraped_finetuned_model")
tokenizer.save_pretrained("./scraped_finetuned_model")
print("Eğitilen model kaydedildi.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

# Kazımak istediğimiz Wikipedia makalesinin URL'si
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"

print("==================================================")
print("       WEB SCRAPING (VERİ KAZIMA) BAŞLATILDI     ")
print("==================================================")
print(f"[ADIM 1] Hedef web adresi belirlendi:\n         -> {url}\n")

# Tarayıcı gibi davranmak için User-Agent başlığı tanımlıyoruz
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}

print("[ADIM 2] Wikipedia sunucusuna bağlantı isteği gönderiliyor...")

try:
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        print(f"[BAŞARILI] Bağlantı sağlandı! (HTTP Durum Kodu: {response.status_code})")
        print("[ADIM 3] Web sayfasının ham HTML kodları bilgisayara indirildi.")
        print("         -> HTML kodları BeautifulSoup ile analiz edilmeye (parse) başlanıyor...")

        # HTML kodlarını çözümlüyoruz
        soup = BeautifulSoup(response.content, 'html.parser')

        print("\n[ADIM 4] Sayfa içindeki gereksiz alanlar (reklam, menü vb.) ayıklanıyor...")
        print("         -> Sadece ana makale gövdesi (id='mw-content-text') aranıyor...")

        content_div = soup.find(id='mw-content-text')

        if content_div:
            print("[BAŞARILI] Makale ana gövdesi tespit edildi.")
            print("         -> Metin paragrafları (<p> etiketleri) toplanıyor...")

            paragraphs = content_div.find_all('p')
            print(f"[BİLGİ] Sayfada toplam {len(paragraphs)} adet ham paragraf bulundu.")

            cleaned_paragraphs = []
            print("\n[ADIM 5] Metin temizleme işlemleri uygulanıyor...")
            print("         -> Paragraflardaki '[1]', '[2]' gibi kaynak linkleri siliniyor...")
            print("         -> Çok kısa veya boş satırlar filtreleniyor...")

            for index, p in enumerate(paragraphs):
                raw_text = p.get_text().strip()

                # Regex (Düzenli İfadeler) ile [1], [2] gibi kaynakça numaralarını temizliyoruz
                text = re.sub(r'\[\d+\]', '', raw_text)
                text = re.sub(r'\[citation needed\]', '', text)

                # En az 40 karakter uzunluğundaki anlamlı paragrafları listeye ekliyoruz
                if len(text) > 40:
                    cleaned_paragraphs.append(text)

            print(f"[BAŞARILI] Temizleme bitti. {len(paragraphs)} paragraftan {len(cleaned_paragraphs)} tanesi eğitime uygun bulundu.")

            # Kullanıcıya ne çektiğimizi göstermek için bir önizleme yazdırıyoruz
            if cleaned_paragraphs:
                print("\n--------------------------------------------------")
                print("     KAZINAN METİNDEN ÖNİZLEME (İLK PARAGRAF)      ")
                print("--------------------------------------------------")
                print(cleaned_paragraphs[0])
                print("--------------------------------------------------\n")

            # Tüm temizlenmiş paragrafları aralarında boşluk bırakarak tek bir metin haline getiriyoruz
            final_text = "\n\n".join(cleaned_paragraphs)

            print("[ADIM 6] Temizlenen veriler dosyaya yazılıyor...")
            file_name = "scraped_wikipedia.txt"
            with open(file_name, "w", encoding="utf-8") as f:
                f.write(final_text)

            print(f"[TAMAMLANDI] Veriler '{file_name}' adıyla başarıyla kaydedildi!")
            print(f"[ÖZET] Toplam Kaydedilen Karakter Sayısı: {len(final_text)}")
            print("==================================================")

        else:
            print("[HATA] Makale gövdesi (mw-content-text) bulunamadı. Wikipedia sayfa yapısı değişmiş olabilir.")
    else:
        print(f"[HATA] Sunucu sayfayı vermeyi reddetti veya sayfa bulunamadı. (Hata Kodu: {response.status_code})")

except Exception as e:
    print(f"[SİSTEM HATASI] İşlem sırasında beklenmeyen bir hata oluştu: {e}")

       WEB SCRAPING (VERİ KAZIMA) BAŞLATILDI     
[ADIM 1] Hedef web adresi belirlendi:
         -> https://en.wikipedia.org/wiki/Artificial_intelligence

[ADIM 2] Wikipedia sunucusuna bağlantı isteği gönderiliyor...
[BAŞARILI] Bağlantı sağlandı! (HTTP Durum Kodu: 200)
[ADIM 3] Web sayfasının ham HTML kodları bilgisayara indirildi.
         -> HTML kodları BeautifulSoup ile analiz edilmeye (parse) başlanıyor...

[ADIM 4] Sayfa içindeki gereksiz alanlar (reklam, menü vb.) ayıklanıyor...
         -> Sadece ana makale gövdesi (id='mw-content-text') aranıyor...
[BAŞARILI] Makale ana gövdesi tespit edildi.
         -> Metin paragrafları (<p> etiketleri) toplanıyor...
[BİLGİ] Sayfada toplam 182 adet ham paragraf bulundu.

[ADIM 5] Metin temizleme işlemleri uygulanıyor...
         -> Paragraflardaki '[1]', '[2]' gibi kaynak linkleri siliniyor...
         -> Çok kısa veya boş satırlar filtreleniyor...
[BAŞARILI] Temizleme bitti. 182 paragraftan 181 tanesi eğitime uygun bulundu.

--------------

In [ ]:
!pip install -q trl peft transformers datasets accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.7 MB/s eta 0:00:00


In [ ]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
import os
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig

print("==================================================")
print("      LLM İNCE AYAR (FINE-TUNING) BAŞLATILDI     ")
print("==================================================")

# --------------------------------------------------
# ADIM 1: Kazınan Veri Dosyasını Okuma ve Hazırlama
# --------------------------------------------------
file_name = "scraped_wikipedia.txt"
print(f"[ADIM 1] '{file_name}' dosyası aranıyor...")

if not os.path.exists(file_name):
    raise FileNotFoundError(f"[HATA] '{file_name}' dosyası bulunamadı! Lütfen önce kazıma (scraping) kodunu çalıştırın.")

with open(file_name, "r", encoding="utf-8") as f:
    # Boş olmayan paragrafları listeye alıyoruz
    paragraphs = [line.strip() for line in f.read().split("\n\n") if line.strip()]

print(f"[BAŞARILI] Dosya okundu. Toplam {len(paragraphs)} adet paragraf eğitim için yükleniyor.")

# Hugging Face Dataset formatına dönüştürme
dataset = Dataset.from_dict({"text": paragraphs})
print("[BİLGİ] Veri seti Hugging Face formatına dönüştürüldü.")


# --------------------------------------------------
# ADIM 2: Küçük Dil Modelini ve Tokenizer'ı Seçme
# --------------------------------------------------
# Qwen2.5-0.5B, oldukça küçük ve standart bilgisayarlarda/ücretsiz GPU'larda rahatça çalışabilen modern bir modeldir.
model_id = "Qwen/Qwen2.5-0.5B"
print(f"\n[ADIM 2] Temel model seçildi: {model_id}")
print("         -> Tokenizer (Kelime bölücü) internetten indiriliyor...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token  # Modelin boşlukları doldurması için gerekli ayar
print("[BAŞARILI] Tokenizer başarıyla yüklendi.")


# --------------------------------------------------
# ADIM 3: Modeli Belleğe Yükleme
# --------------------------------------------------
print("\n[ADIM 3] Modelin ana ağırlıkları belleğe (RAM/VRAM) yükleniyor...")
print("         -> Bu işlem internet hızınıza bağlı olarak birkaç dakika sürebilir...")

# Eğer bilgisayarınızda Nvidia ekran kartı (GPU) varsa CUDA kullanılacaktır
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[BİLGİ] Hesaplama birimi olarak '{device.upper()}' kullanılacak.")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
print("[BAŞARILI] Ana model belleğe başarıyla yüklendi.")


# --------------------------------------------------
# ADIM 4: LoRA (Hafif Eğitim) Ayarlarını Yapma
# --------------------------------------------------
# LoRA, milyarlarca parametreye sahip modellerin tamamını eğitmek yerine,
# sadece küçük bir kısmını eğiterek bilgisayarımızı yormamamızı sağlayan bir yöntemdir.
print("\n[ADIM 4] LoRA (Hafif Eğitim) parametreleri yapılandırılıyor...")
peft_config = LoraConfig(
    r=8,              # Güncellenecek ağırlık matrisinin boyutu (Küçük olması hafıza dostudur)
    lora_alpha=16,    # Eğitimin modele olan etki gücü katsayısı
    lora_dropout=0.05,# Aşırı öğrenmeyi engellemek için rastgele pasif bırakılacak nöron oranı
    bias="none",
    task_type="CAUSAL_LM"
)
print("[BAŞARILI] LoRA ayarları tamamlandı.")


# --------------------------------------------------
# ADIM 5: Eğitim Parametrelerini Belirleme (SFTConfig)
# --------------------------------------------------
print("\n[ADIM 5] Eğitim motoru ayarlanıyor (SFTConfig kullanılacak)...")

# CPU üzerinde eğitim yapılabilmesi için 'use_cpu=True' eklenmiştir.
training_args = SFTConfig(
    output_dir="./results",
    per_device_train_batch_size=2,      # Tek seferde modele verilecek paragraf sayısı
    gradient_accumulation_steps=4,      # Belleği yormamak için gradyan biriktirme adımı
    learning_rate=2e-4,                 # Öğrenme hızı
    logging_steps=5,                    # Her 5 adımda bir ekrana ilerleme durumunu yazdır
    max_steps=20,                       # Toplam eğitim adım sayısı
    fp16=(device == "cuda"),            # GPU varsa 16-bit hızlandırmayı aç, CPU için False olur
    bf16=False,                         # CPU eğitimi için bf16 kapatıldı
    save_strategy="no",                 # Ara adımlarda diske büyük dosyalar kaydetme

    # CPU eğitimi için zorunlu parametre:
    use_cpu=(device == "cpu"),          # Hesaplama birimi CPU ise True, GPU ise False olur

    # SFT'ye özel parametreler:
    dataset_text_field="text",
    max_length=512
)
print("[BAŞARILI] SFTConfig ayarları tamamlandı.")


# --------------------------------------------------
# ADIM 6: Eğitimi Başlatma
# --------------------------------------------------
print("\n[ADIM 6] SFTTrainer (İnce Ayar Eğitmeni) oluşturuluyor...")
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
    peft_config=peft_config
)

print("\n==================================================")
print("             EĞİTİM DÖNGÜSÜ BAŞLIYOR              ")
print("==================================================")
print("Sistem verileri okuyor ve modeli eğitmeye başlıyor. Lütfen bekleyin...")

trainer.train()

print("\n==================================================")
print("             EĞİTİM BAŞARIYLA BİTTİ               ")
print("==================================================")


# --------------------------------------------------
# ADIM 7: Yeni Modeli Kaydetme
# --------------------------------------------------
save_directory = "./wikipedia_trained_model"
print(f"\n[ADIM 7] Eğitilen yeni model '{save_directory}' klasörüne kaydediliyor...")

trainer.model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"[TAMAMLANDI] Model başarıyla kaydedildi. Artık bu modeli kendi verilerinizle test edebilirsiniz!")
print("==================================================")

      LLM İNCE AYAR (FINE-TUNING) BAŞLATILDI     
[ADIM 1] 'scraped_wikipedia.txt' dosyası aranıyor...
[BAŞARILI] Dosya okundu. Toplam 181 adet paragraf eğitim için yükleniyor.
[BİLGİ] Veri seti Hugging Face formatına dönüştürüldü.

[ADIM 2] Temel model seçildi: Qwen/Qwen2.5-0.5B
         -> Tokenizer (Kelime bölücü) internetten indiriliyor...
[BAŞARILI] Tokenizer başarıyla yüklendi.

[ADIM 3] Modelin ana ağırlıkları belleğe (RAM/VRAM) yükleniyor...
         -> Bu işlem internet hızınıza bağlı olarak birkaç dakika sürebilir...
[BİLGİ] Hesaplama birimi olarak 'CUDA' kullanılacak.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/tmp/ipykernel_698/571883459.py:85: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


[BAŞARILI] Ana model belleğe başarıyla yüklendi.

[ADIM 4] LoRA (Hafif Eğitim) parametreleri yapılandırılıyor...
[BAŞARILI] LoRA ayarları tamamlandı.

[ADIM 5] Eğitim motoru ayarlanıyor (SFTConfig kullanılacak)...
[BAŞARILI] SFTConfig ayarları tamamlandı.

[ADIM 6] SFTTrainer (İnce Ayar Eğitmeni) oluşturuluyor...


Adding EOS to train dataset:   0%|          | 0/181 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/181 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



             EĞİTİM DÖNGÜSÜ BAŞLIYOR              
Sistem verileri okuyor ve modeli eğitmeye başlıyor. Lütfen bekleyin...


Step,Training Loss
5,2.780100
10,2.973422
15,2.830340
20,2.824890



             EĞİTİM BAŞARIYLA BİTTİ               

[ADIM 7] Eğitilen yeni model './wikipedia_trained_model' klasörüne kaydediliyor...
[TAMAMLANDI] Model başarıyla kaydedildi. Artık bu modeli kendi verilerinizle test edebilirsiniz!


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("==================================================")
print("          EĞİTİLMİŞ MODEL TEST PANELİ             ")
print("==================================================")

# 1. Model ve Kayıt Klasörü Bilgileri
base_model_id = "Qwen/Qwen2.5-0.5B"
adapter_dir = "./wikipedia_trained_model"

# 2. Donanım Kontrolü (GPU mu CPU mu?)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[ADIM 1] Test için kullanılacak donanım belirlendi: '{device.upper()}'")

# 3. Kelime Bölücünün (Tokenizer) Yüklenmesi
print("[ADIM 2] Kelime bölücü (Tokenizer) dosyaları yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

# 4. Temel Modelin Belleğe Yüklenmesi
print(f"[ADIM 3] Temel model ({base_model_id}) belleğe alınıyor...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

# 5. Eğitilen LoRA Katmanının Temel Modelle Birleştirilmesi
print("[ADIM 4] Sizin eğittiğiniz yeni bilgi katmanı (LoRA) modele ekleniyor...")
model = PeftModel.from_pretrained(base_model, adapter_dir)

# Modeli test/değerlendirme moduna alıyoruz (Eğitim modunu kapatır)
model.eval()
print("[BAŞARILI] Modeliniz başarıyla yüklendi ve test edilmeye hazır!")
print("==================================================\n")


# -----------------------------------------------------------------
# TEST ALANI: Aşağıdaki başlangıç cümlesini dilediğiniz gibi değiştirebilirsiniz
# -----------------------------------------------------------------

# Wikipedia makalesine uygun bir başlangıç cümlesi (Prompt) yazıyoruz:
prompt = "One of the main challenges in artificial intelligence is"


print(f"Yapay Zekaya Verilen Başlangıç: '{prompt}'")
print("Yapay zeka cümleyi tamamlıyor, lütfen bekleyin...\n")

# Cümleyi sayılara (token) çeviriyoruz ve cihazımıza (GPU/CPU) gönderiyoruz
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Modelin yeni kelimeler türeterek devamını yazmasını sağlıyoruz
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,      # Modelin en fazla kaç yeni kelime üreteceğini belirler
        temperature=0.7,         # Yaratıcılık katsayısı (0.1 daha tutarlı/ezbere, 1.0 daha yaratıcı)
        do_sample=True,          # Doğal konuşma akışı için örneklemeyi açar
        top_k=50,                # En olası ilk 50 kelime arasından seçim yapmasını sağlar
        top_p=0.9,               # Kelime seçiminde çeşitlilik dengesi kurar
        repetition_penalty=1.2   # Aynı kelimeleri sürekli tekrar etmesini engeller
    )

# Sayısal çıktıları tekrar insanların okuyabileceği metne çeviriyoruz
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("==================================================")
print("             YAPAY ZEKANIN YAZDIĞI METİN          ")
print("==================================================")
print(generated_text)
print("==================================================")

          EĞİTİLMİŞ MODEL TEST PANELİ             
[ADIM 1] Test için kullanılacak donanım belirlendi: 'CUDA'
[ADIM 2] Kelime bölücü (Tokenizer) dosyaları yükleniyor...
[ADIM 3] Temel model (Qwen/Qwen2.5-0.5B) belleğe alınıyor...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[ADIM 4] Sizin eğittiğiniz yeni bilgi katmanı (LoRA) modele ekleniyor...


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[BAŞARILI] Modeliniz başarıyla yüklendi ve test edilmeye hazır!

Yapay Zekaya Verilen Başlangıç: 'One of the main challenges in artificial intelligence is'
Yapay zeka cümleyi tamamlıyor, lütfen bekleyin...

             YAPAY ZEKANIN YAZDIĞI METİN          
One of the main challenges in artificial intelligence is to find ways for people who are deaf or hard-of-hearing, as well as blind and visually impaired individuals. The AI industry has a long way to go before it can be used effectively by these groups.
In this video from the 2017 International Conference on Artificial Intelligence Education (ICAE), two researchers explain what makes a good chatbot: “You have got to build your conversation with them so that they understand you better.”
The solution? A new language called Natural Language Processing (


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("==================================================")
print("          EĞİTİLMİŞ MODEL TEST PANELİ             ")
print("==================================================")

# 1. Model ve Kayıt Klasörü Bilgileri
base_model_id = "Qwen/Qwen2.5-0.5B"
adapter_dir = "./wikipedia_trained_model"

# 2. Donanım Kontrolü (GPU mu CPU mu?)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[ADIM 1] Test için kullanılacak donanım belirlendi: '{device.upper()}'")

# 3. Kelime Bölücünün (Tokenizer) Yüklenmesi
print("[ADIM 2] Kelime bölücü (Tokenizer) dosyaları yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

# 4. Temel Modelin Belleğe Yüklenmesi
print(f"[ADIM 3] Temel model ({base_model_id}) belleğe alınıyor...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

# 5. Eğitilen LoRA Katmanının Temel Modelle Birleştirilmesi
print("[ADIM 4] Sizin eğittiğiniz yeni bilgi katmanı (LoRA) modele ekleniyor...")
model = PeftModel.from_pretrained(base_model, adapter_dir)

# Modeli test/değerlendirme moduna alıyoruz (Eğitim modunu kapatır)
model.eval()
print("[BAŞARILI] Modeliniz başarıyla yüklendi ve test edilmeye hazır!")
print("==================================================\n")

# -----------------------------------------------------------------
# İNTERAKTİF TEST ALANI: Kullanıcıdan giriş alma
# -----------------------------------------------------------------

while True: # Kullanıcı çıkmak isteyene kadar döngü devam eder
    print("--------------------------------------------------")
    user_prompt = input("Lütfen bir başlangıç cümlesi girin (Çıkmak için 'q' yazın): \n")

    if user_prompt.lower() == 'q':
        print("\nTest sonlandırılıyor. Güle güle!")
        break

    if not user_prompt.strip(): # Boş giriş kontrolü
        print("Boş bir cümle girdiniz. Lütfen geçerli bir metin girin.")
        continue

    print("\nYapay zeka cümleyi tamamlıyor, lütfen bekleyin...\n")

    # Cümleyi sayılara (token) çeviriyoruz ve cihazımıza (GPU/CPU) gönderiyoruz
    inputs = tokenizer(user_prompt, return_tensors="pt").to(device)

    # Modelin yeni kelimeler türeterek devamını yazmasını sağlıyoruz
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,      # Modelin en fazla kaç yeni kelime üreteceğini belirler
            temperature=0.7,         # Yaratıcılık katsayısı (0.1 daha tutarlı/ezbere, 1.0 daha yaratıcı)
            do_sample=True,          # Doğal konuşma akışı için örneklemeyi açar
            top_k=50,                # En olası ilk 50 kelime arasından seçim yapmasını sağlar
            top_p=0.9,               # Kelime seçiminde çeşitlilik dengesi kurar
            repetition_penalty=1.2   # Aynı kelimeleri sürekli tekrar etmesini engeller
        )

    # Sayısal çıktıları tekrar insanların okuyabileceği metne çeviriyoruz
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("==================================================")
    print("             YAPAY ZEKANIN YAZDIĞI METİN          ")
    print("==================================================")
    print(generated_text)
    print("==================================================\n")

          EĞİTİLMİŞ MODEL TEST PANELİ             
[ADIM 1] Test için kullanılacak donanım belirlendi: 'CUDA'
[ADIM 2] Kelime bölücü (Tokenizer) dosyaları yükleniyor...
[ADIM 3] Temel model (Qwen/Qwen2.5-0.5B) belleğe alınıyor...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[ADIM 4] Sizin eğittiğiniz yeni bilgi katmanı (LoRA) modele ekleniyor...
[BAŞARILI] Modeliniz başarıyla yüklendi ve test edilmeye hazır!

--------------------------------------------------
Lütfen bir başlangıç cümlesi girin (Çıkmak için 'q' yazın): 
The history of artificial intelligence began with


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Yapay zeka cümleyi tamamlıyor, lütfen bekleyin...

             YAPAY ZEKANIN YAZDIĞI METİN          
The history of artificial intelligence began with the 1950s, when Alan Turing proposed a machine that could think like humans. Since then, researchers have made significant progress in developing algorithms and systems to make machines learn from experience.
In this article we will discuss how AI works: what is it? What are its applications?
What Is Artificial Intelligence
Artificial intelligence (AI) refers to computers or software designed to perform tasks that require human-like reasoning skills. These include image recognition, speech-to-text conversion, decision-making processes,

--------------------------------------------------
Lütfen bir başlangıç cümlesi girin (Çıkmak için 'q' yazın): 
q

Test sonlandırılıyor. Güle güle!
